# Read the Floor — Colab pipeline

Skalski's open-source vision pipeline extracts coordinates from a clip; **our brain** reads
the floor (who guards whom + Beaten Index) and renders the overlay.

```
video --> RF-DETR --> SAM2 --> SigLIP/K-means --> SmolVLM2 --> court homography
      --> [skalski adapter] --> tracking table --> brain --> render
```

**Runtime -> Change runtime type -> GPU** before running.


## 1. Install the open-source stack


In [1]:
!pip -q install rfdetr supervision sam2 transformers umap-learn scikit-learn imageio matplotlib pandas scipy
!pip -q install git+https://github.com/roboflow/sports.git


## 2. Get our modules (schema / brain / render / adapters)
Clone the portfolio repo, or upload the `read_the_floor/` folder to Colab.


In [2]:
# option A: clone (if the repo is pushed & public)
# !git clone https://github.com/aldemirkonuk/Portfolio.git && %cd Portfolio/read_the_floor
import sys; sys.path.insert(0,'/content/read_the_floor')  # option B: uploaded folder
import schema, brain, render
from adapters import skalski


## 3. Skalski's vision pipeline  (adapt from his notebook)

Use the cells from **basketball-ai-how-to-detect-track-and-identify-basketball-players**
(linked in his video) to load the models and process your clip. By the end of his flow
you should have, **per frame**, these four objects — that's all our adapter needs:

- `players`     : `supervision.Detections`  (RF-DETR boxes + SAM2 `tracker_id`)
- `ball`        : `supervision.Detections`  (the ball; may be empty)
- `transformer` : `sports` `ViewTransformer`  (image px -> court ft, from court keypoints)
- `team_by_id`  : `{tracker_id: 0/1}`  (SigLIP + K-means)
- `number_by_id`: `{tracker_id: '23'}`  (SmolVLM2 / ResNet, optional)


In [ ]:
# ---- SKELETON: fill from Skalski's notebook ----
from rfdetr import RFDETRBase            # detector
import supervision as sv
# from sam2... import build tracker      # SAM2 tracker
# from sports.common.view import ViewTransformer

VIDEO = '/content/clip.mp4'              # your possession clip
detector = RFDETRBase()                  # load weights per his notebook

# court homography: fit a ViewTransformer from the court-keypoint model each frame.
# team_by_id: SigLIP embeddings of player crops -> UMAP -> KMeans(2).
# number_by_id: SmolVLM2/ResNet on jersey crops, voted across frames.
#
# Produce `frames_out`: a list, one entry per frame, each = (players, ball,
# transformer, team_by_id, number_by_id). See his notebook for the exact calls.


## 4. Adapt -> tracking table  (our interface)


In [ ]:
per_frame = []
for i,(players, ball, transformer, team_by_id, number_by_id) in enumerate(frames_out):
    per_frame.append(skalski.frame_to_rows(i, players, ball, transformer,
                                           team_by_id, number_by_id))
df = skalski.build_table(per_frame, smooth=True)   # exponential track smoothing
schema.validate(df)
df.head()


## 5. Run the brain + render the overlay


In [ ]:
results, off_team, hoop = brain.run(df)
render.render_gif(results, df, '/content/matchup_engine.gif')
from IPython.display import Image; Image('/content/matchup_engine.gif')


## Next
- Render back onto the **footage** via the inverse homography (court -> image px).
- Swap exponential smoothing for a Kalman filter.
- Stack the Value Surface (EPV) + Risk on top of the same table.
